# 1. Data Loading and Initial Setup

1.  Load and clean the raw data, dropping the 6 rows missing horsepower data
2.  Encode data:
    - Map `binaryClass` P -> 1, N -> 0
    - One-hot encode `origin` (nominal, drop_first=False)
    - `cylinders` kept as int (ordinal, engine size related)
    - `model` kept as int (ordinal, year progression)
    - Continuous features unchanged
3.  Create a single, hold-out test set
4.  From the main training set, generate multiple training datasets with varying **imbalance ratios (IR)** by undersampling the majority class
5.  For each IR, create **multiple repetitions** with different random samples of the minority class
6.  For each IR and repetition, create a size-matched **control dataset** with the original class ratio
7.  Preprocess and save all generated datasets


# Dataset Configuration

In [1]:
DATASET_NAME = "auto_mpg"

print(f"Dataset: {DATASET_NAME}")

Dataset: auto_mpg


In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from pathlib import Path
import sys

sys.path.append(str(Path("../../../../").resolve()))
from config.config import get_config

config = get_config()

DATASET_NAME = "auto_mpg"
TARGET_FEATURE = "binaryClass"
CLASS_P = 1
CLASS_N = 0
RANDOM_STATE = 42
IMBALANCE_RATIOS = config.experiment.imbalance_ratios
N_REPETITIONS = config.experiment.n_repetitions

RAW_PATH = Path(f"../../../../data/raw/{DATASET_NAME}.csv")
PROCESSED_PATH = Path(f"../../../../data/processed/{DATASET_NAME}/")

RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  • Dataset: {DATASET_NAME}")
print(f"  • Raw data path: {RAW_PATH}")
print(f"  • Processed data path: {PROCESSED_PATH}")
print(f"  • Target feature: {TARGET_FEATURE}")
print(f"\nFetching and loading the raw dataset...")

# load dataset
dataset = fetch_openml(data_id=831, as_frame=True)
df_raw = pd.concat([dataset.data, dataset.target], axis=1)
df_raw.to_csv(RAW_PATH, index=False)
print(f"Raw dataset loaded and saved. Shape: {df_raw.shape}")

ModuleNotFoundError: No module named 'config.config'

# 2. Data Cleaning and Encoding

As determined in the EDA, we will drop the entries without horsepower values and encode categorical features and the target class.

In [ ]:
clean_df = df_raw.copy()

# drop missing horsepower rows
clean_df['horsepower'] = pd.to_numeric(clean_df['horsepower'], errors='coerce')
clean_df = clean_df.dropna(subset=['horsepower'])
print(f"Shape after dropping missing horsepower: {clean_df.shape}")

# encode target
clean_df['binaryClass'] = clean_df['binaryClass'].map({'P': 1, 'N': 0})

# one-hot encode categorical feature origin
clean_df = pd.get_dummies(clean_df, columns=['origin'], drop_first=False)

# convert bool columns to int
bool_cols = clean_df.select_dtypes(include='bool').columns
clean_df[bool_cols] = clean_df[bool_cols].astype(int)
clean_df['model'] = clean_df['model'].astype(int)
clean_df['binaryClass'] = clean_df['binaryClass'].astype(int)

# check binaryClass worked
print(clean_df['binaryClass'].value_counts())

print(f"\nFinal shape: {clean_df.shape}")
print(f"\nColumns: {clean_df.columns.tolist()}")
print(f"\nMissing values: {clean_df.isnull().sum().sum()}")
print(f"\ndtypes:\n{clean_df.dtypes}")

# 3. Confirming Majority and Minority Classes
#
For our imbalance experiments, we need to clearly define the majority and minority classes.
- **Class 0 (Minority):** Negative
- **Class 1 (Majority):** Positive


In [ ]:
print("Target variable distribution:")
print(clean_df[TARGET_FEATURE].value_counts())


# 4. Create a Hold-Out Test Set

We perform a one-time stratified split to create a final test set. All experimental datasets will be generated from the `train_full_df`.


In [ ]:
X = clean_df.drop(TARGET_FEATURE, axis=1)
y = clean_df[[TARGET_FEATURE]]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=config.experiment.test_size,
    random_state=RANDOM_STATE,
    stratify=y
)

train_full_df = pd.concat([X_train_full, y_train_full], axis=1)

print(f"Full training set shape: {train_full_df.shape}")
print(f"Hold-out test set shape: {X_test.shape}")

# 5. Generate Imbalanced and Control Datasets with Multiple Repetitions

For each IR, we now create multiple repetitions by sampling 
different subsets of the minority class. This allows us to test whether methods 
work reliably across different minority class samples.

1.  Start with the **full majority class** ('Positive').
2.  **Undersample the minority class** ('Negative') to achieve the desired Imbalance Ratio (IR).
3.  **Repeat this sampling N_REPETITIONS times** with different random seeds.
4.  Create a size-matched **control dataset** for each IR and repetition.



  * **Methodology:** "To generate datasets with varying degrees of class imbalance, the majority class was held constant at 165 samples while the minority class was progressively undersampled to achieve imbalance ratios from 5:1 to 100:1. It should be noted that this approach intrinsically links a higher imbalance ratio with a smaller number of minority class samples."
  * **Discussion:** When interpreting your results, we can't claim that the degradation in synthetic data quality is *only* due to the imbalance ratio. 

In [ ]:
negative_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_N]
positive_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_P]
n_minority_available = len(negative_df)
n_majority_available = len(positive_df)

print(f"\nFull training set composition: {n_majority_available} majority (Positive), {n_minority_available} minority (Negative).")
print(f"\nGenerating datasets with {N_REPETITIONS} repetitions per imbalance ratio...")

generated_datasets = {}

for ir in IMBALANCE_RATIOS:
    print(f"\n")
    print(f"Processing Imbalance Ratio (IR) = {ir}:1")
    print(f"\n")
    
    for rep_id in range(1, N_REPETITIONS + 1):
        print(f"\n  Repetition {rep_id}/{N_REPETITIONS}")
        
        # Use different random seed for each repetition
        rep_seed = RANDOM_STATE + (ir * 1000) + rep_id
        
        if ir == 1:
            majority_undersampled = resample(
                positive_df,
                replace=False,
                n_samples=n_minority_available, 
                random_state=rep_seed 
            )
            imbalanced_df = pd.concat([majority_undersampled, negative_df])
            
        else:
            majority_full_set = positive_df
            
            n_minority_imbalanced = int(n_majority_available / ir)

            if n_minority_imbalanced > n_minority_available:
                print(f"    SKIPPING: Cannot create {ir}:1 ratio as it requires more minority samples than available.")
                continue
            if n_minority_imbalanced < 1:
                print(f"    SKIPPING: Ratio {ir}:1 results in zero minority samples.")
                continue

            minority_undersampled = resample(
                negative_df,
                replace=False,
                n_samples=n_minority_imbalanced,
                random_state=rep_seed 
            )

            imbalanced_df = pd.concat([majority_full_set, minority_undersampled])

        total_size = len(imbalanced_df)
        
        dataset_key = f'imbalanced_ir_{ir}_rep{rep_id}'
        generated_datasets[dataset_key] = imbalanced_df
        
        n_maj = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_P])
        n_min = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_N])
        print(f"    Imbalanced set created: {total_size} samples ({n_maj} majority, {n_min} minority)")

        if total_size >= len(train_full_df):
            control_df = train_full_df.copy()
        else:
            control_df, _ = train_test_split(
                train_full_df,
                train_size=total_size,
                random_state=rep_seed,  
                stratify=train_full_df[TARGET_FEATURE]
            )
        
        control_key = f'control_ir_{ir}_rep{rep_id}'
        generated_datasets[control_key] = control_df
        print(f"    Control set created:      {len(control_df)} samples (original class ratio)")

print(f"Dataset generation complete!")
print(f"Total datasets created: {len(generated_datasets)}")
print(f"  - Imbalanced: {len([k for k in generated_datasets.keys() if 'imbalanced' in k])}")
print(f"  - Control: {len([k for k in generated_datasets.keys() if 'control' in k])}")

# 6. Preprocessing and Saving All Datasets

We fit the scaler **once** on the full training data. Then, we transform all generated training sets and the hold-out test set using this single, consistent scaler.


In [ ]:
FEATURES_TO_SCALE = [col for col in clean_df.columns 
                     if col != TARGET_FEATURE]

scaler = StandardScaler()
scaler.fit(X_train_full[FEATURES_TO_SCALE])

print("Scaling and saving datasets...\n")

for name, df in generated_datasets.items():
    X_temp = df.drop(columns=[TARGET_FEATURE])
    y_temp = df[[TARGET_FEATURE]]

    X_processed = X_temp.copy()
    X_processed[FEATURES_TO_SCALE] = scaler.transform(X_temp[FEATURES_TO_SCALE])
    
    final_df = X_processed.reset_index(drop=True)
    final_df[TARGET_FEATURE] = y_temp.reset_index(drop=True)
    
    save_path = PROCESSED_PATH / f"train_{name}.csv"
    final_df.to_csv(save_path, index=False)
    print(f"Saved: {save_path.name}")

X_test_processed = X_test.copy()
X_test_processed[FEATURES_TO_SCALE] = scaler.transform(X_test[FEATURES_TO_SCALE])

test_df = X_test_processed.reset_index(drop=True)
test_df[TARGET_FEATURE] = y_test.reset_index(drop=True)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print(f"\nSaved test set: test.csv")
print(f"Total training files: {len(generated_datasets)}")

In [ ]:
import json
from datetime import datetime

# Save metadata about this dataset processing
metadata = {
    "dataset_name": DATASET_NAME,
    "target_feature": TARGET_FEATURE,
    "processing_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_train_files": len(generated_datasets),
    "imbalance_ratios": IMBALANCE_RATIOS,
    "n_repetitions": N_REPETITIONS,
    "random_state": RANDOM_STATE,
    "test_size": len(test_df),
    "features": FEATURES_TO_SCALE,
    "onehot_encoded_features": ["origin"],
    "ordinal_features": ["model", "cylinders"]
}

metadata_path = PROCESSED_PATH / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to: {metadata_path}")
print("\nProcessing complete! All datasets are ready for experiments.")